# 02 — Fit calibrated fraud score providers

Begin with LightGBM. The first run compares the control features against one causal feature block at a time using validation cost and PR-AUC. Run notebook 03 and validate the LightGBM-plus-bandit pipeline before including XGBoost and CatBoost. Each model is an alternative fixed score provider, not a bandit action.

In [ ]:
from pathlib import Path
import os, sys
candidate_src = [Path(os.environ.get('FRAUDLAB_SRC', 'src')), Path('/kaggle/input/ieee-cis-fraud-code/src')]
for src in candidate_src:
    if (src / 'fraudlab').is_dir():
        sys.path.insert(0, str(src.resolve()))
        break
else:
    raise FileNotFoundError('Set FRAUDLAB_SRC to the src directory from this project.')

from fraudlab.config import ExperimentConfig
from fraudlab.pipeline import train_models_from_artifact
from fraudlab.paths import resolve_artifact_dir

FEATURE_ROOT = resolve_artifact_dir('features/ieee_cis_features.parquet')
OUTPUT_ROOT = Path(os.environ.get('IEEE_CIS_OUTPUT_DIR', '/kaggle/working/artifacts' if Path('/kaggle/working').exists() else 'artifacts'))
FEATURE_PATH = FEATURE_ROOT / 'features' / 'ieee_cis_features.parquet'
OUTPUT_DIR = OUTPUT_ROOT / 'experiments'
MODEL_NAMES = ['lightgbm']  # After Notebook 03 passes, set to ['xgboost', 'catboost']
RUN_FEATURE_ABLATION = True  # Keep True for the first LightGBM run; False for later model runs
config = ExperimentConfig()
print('Feature artifact:', FEATURE_PATH)
print('Models this run:', MODEL_NAMES)

In [ ]:
training_summary = train_models_from_artifact(
    feature_path=FEATURE_PATH,
    output_dir=OUTPUT_DIR,
    model_names=MODEL_NAMES,
    config=config,
    run_feature_ablation=RUN_FEATURE_ABLATION,
)
training_summary